# Blockchain Case Study — Delivery Risk

> **Will this feature stay within budget if the team runs late?**

This notebook answers one question: **what does it actually cost to deliver a feature when sprints run over plan?**

It is a pure **cost and schedule risk** view. Business value and business value are covered in other notebooks.

### How this fits into the learning path

| Notebook | What it answers |
|---|---|
| 05 — Risk Assessment | How likely is the feature to deliver its expected business value? |
| **06 — Delivery Risk** | **How much will it cost if the team runs late? Can the budget hold?** |
| 03 — Capital Budgeting | Is the investment worthwhile over multiple years? (NPV, IRR) |

### What this notebook simulates

The simulation runs each feature through thousands of scenarios. In each scenario, the team delivers the feature in a randomly drawn number of sprints — sometimes on time, sometimes late. The simulation then calculates the cost for that scenario.

From all scenarios together you can see:
- How often delivery runs over the planned sprint count
- How much more it costs in a bad case
- Which features are most likely to blow the budget

### What each section covers

| Section | Question |
|---|---|
| 1 — Setup | What are the assumptions? How many scenarios are we running? |
| 2 — Sprint Plan | How many sprints does each feature need? Does everything fit in one quarter? |
| 3 — Delay Analysis | How much later than planned could delivery be? |
| 4 — Cost Risk | How much more expensive could each feature get? |
| 5 — Budget Fit | Which features are safe to fund? Which combination keeps cost risk lowest? |

---

*Previous: [05-blockchain-case-study-risk.ipynb](05-blockchain-case-study-risk.ipynb)*

In the previous notebook we measured what happens to business value when risks hit. Now we ask: what does it cost if delivery fails? Value loss and cost overrun are two sides of the same coin.


Sunk costs are money already spent and not recoverable. Track them to set review gates — if cancelling after Sprint 4 costs 120k, consider a decision point at Sprint 3.


The business value floor measures the floor of your business value; LaR (Loss at Risk) measures the ceiling of your potential loss. They look at the same distribution from opposite sides.


## 1) Setup — Run the Delivery Simulation

**Question:** Are the delivery-risk assumptions clear and realistic?

The code below loads the scenario configuration and runs the full delivery simulation.
`delivery_results` is computed **once here** — every table and chart in this notebook reads from it.

### What the simulation does

Each feature has a planned number of sprints based on its estimated development weeks.
The simulation runs thousands of scenarios. In each scenario, the actual sprint count is drawn randomly — sometimes equal to plan, sometimes more.

The random draw uses a **lognormal distribution**: most scenarios finish close to plan, but a small number run significantly late. This reflects real teams — rarely early, occasionally very late.

### How planned sprints are calculated

$$S_{\mathrm{plan}} = \left\lceil \frac{W}{L} \right\rceil$$

$W$ = estimated development weeks · $L$ = sprint length in weeks.

### How actual sprints are simulated

The actual sprint count follows a truncated lognormal distribution, moment-matched so that:

- The **expected value** equals the planned sprint count: $E[X] = S_{\mathrm{plan}}$
- The **standard deviation** equals planned sprints × the configured uncertainty: $\mathrm{Std}[X] = S_{\mathrm{plan}} \cdot \sigma_{\mathrm{delay}}$

The distribution is truncated at $S_{\mathrm{plan}} \cdot c_{\mathrm{ceil}}$ (the sprint ceiling from config). The draw is rounded up to a whole sprint number.

Log-space parameters derived by moment-matching:
$\sigma_{\log} = \sqrt{\ln(1 + \sigma_{\mathrm{delay}}^2)},\quad \mu_{\log} = \ln(S_{\mathrm{plan}}) - \tfrac{1}{2}\sigma_{\log}^2$

### How delivery cost is calculated (per scenario)

$$C_{\mathrm{actual}} = S_{\mathrm{actual}} \times L \times \frac{C_{\mathrm{dev}}}{W}$$

More sprints → more team-weeks → more cost.
A cancelled feature still incurs the cost of sprints run up to the cancellation trigger point (sunk cost).

### Where do the delivery parameters come from?

The delay model uses a **lognormal distribution**: most projects finish close to plan, but a small number run significantly late. This shape matches empirical findings from IT project research:

- [**Standish Group CHAOS Reports**](https://www.infoq.com/articles/standish-chaos-2015/) consistently show that only ~30% of IT projects finish on time and on budget. The distribution of overruns is right-skewed — exactly what lognormal captures.
- [**McKinsey/Oxford research**](https://www.mckinsey.com/capabilities/tech-and-ai/our-insights/delivering-large-scale-it-projects-on-time-on-budget-and-on-value) on large IT projects found that 45% run over budget and 7% run over time, with cost overruns averaging 45% of the planned budget.

The specific delay parameters in this scenario (uncertainty, sprint ceiling, cancellation thresholds) are **configurable estimates** in . They are starting points, not universal truths.

> **Calibrate with your own data.** If your team tracks actual vs. planned sprints, use that history to set the delay uncertainty. Two sprints of team data are more valuable than any industry benchmark.

In [ ]:
from fhs.application import AdvancedPortfolioService
from fhs.notebook import notebook_setup
from fhs.presentation.notebook import COLORS, show
from fhs.presentation.notebook.charts import (
    plot_cost_comparison,
    plot_sprint_delay_grid,
)

In [ ]:
setup = notebook_setup("blockchain")
scenario = setup.scenario
delivery_config = setup.delivery_config

if scenario is None or delivery_config is None:
    raise RuntimeError("Scenario setup could not be initialized")

features = sorted(scenario.features, key=lambda x: x.name)
feature_names = [f.name for f in features]

configured_scenarios = int(scenario.scenarios)
delivery_config = delivery_config.model_copy(update={"scenarios": configured_scenarios})

service = AdvancedPortfolioService.from_scenario(
    scenario, seed=scenario.seed, scenarios=configured_scenarios
)
delivery_results = service.delivery.simulate_risk(
    feature_names, delivery_config=delivery_config, seed=scenario.seed
)

In [ ]:
sprint_length_weeks = float(delivery_config.sprint_length_weeks)
portfolio_weekly_burn = sum(
    float(f.development_cost) / float(f.development_weeks)
    for f in features
    if f.development_weeks
)
portfolio_sprint_burn = portfolio_weekly_burn * sprint_length_weeks

In [ ]:
show.columns(
    show.info_html(
        f"Source: <code>{scenario.config_path}</code><br>"
        f"Budget: <b>EUR {scenario.budget:,.0f}</b> · Features: <b>{len(features)}</b><br>"
        f"Sprint length: <b>{delivery_config.sprint_length_weeks} weeks</b> · "
        f"Quarterly capacity: <b>{delivery_config.quarterly_capacity_sprints} sprints</b>"
    ),
    show.note_html(
        f"Delivery simulation scenarios (from config): <b>{configured_scenarios:,}</b><br>"
        f"Features simulated: <b>{len(delivery_results)}</b>",
        compact=True,
    ),
    min_width="340px",
)
show.delivery_risk_configuration(delivery_config)

## 2) Sprint Plan — Capacity Check

**Question:** Can we deliver all planned features within one quarter?

This section uses only planning inputs — no simulation yet.

### What this checks

Each feature needs a certain number of sprints. Add them all up and compare to the quarterly sprint capacity.

- **Fits:** the team can deliver everything within the quarter on plan.
- **Over capacity:** the team is already overloaded before any delays occur.

This is the **baseline check** — if you are over capacity on paper, the cost risk in Section 4 is understated for a single-quarter delivery window.

> **Product Owner:** Use this to decide whether to descope before the sprint starts.
>
> **Risk Manager:** A capacity overrun here means schedule pressure exists before any uncertainty is added.

In [ ]:
capacity_sprints = delivery_config.quarterly_capacity_sprints

sprint_html, plan_rows = show.sprint_plan_html(features, delivery_config)
total_planned_sprints = sum(int(row[2]) for row in plan_rows)

In [ ]:
show.columns(
    sprint_html,
    show.metrics_html(
        [
            (
                "Quarterly sprint capacity",
                f"{capacity_sprints:.1f} sprints",
                COLORS.primary,
            ),
            (
                "Total planned sprints (all features)",
                f"{total_planned_sprints}",
                COLORS.warning,
            ),
            (
                "Capacity status",
                "Fits"
                if total_planned_sprints <= capacity_sprints
                else "Over capacity",
                COLORS.success
                if total_planned_sprints <= capacity_sprints
                else COLORS.danger,
            ),
        ],
        title="Capacity Check",
    ),
    min_width="320px",
)

## 3) Delay Analysis — How Far Can Delivery Deviate From Plan?

**Question:** In how many scenarios does a feature run over its planned sprint count — and by how much?

### What the table shows

| Column | Meaning |
|---|---|
| **Planned sprints** | Fixed from config — $S_{\mathrm{plan}} = \lceil W / L \rceil$ |
| **P50 actual** | Half of all scenarios finish within this many sprints |
| **P75 actual** | 75% of all scenarios finish within this many sprints |
| **P95 actual** | 95% of all scenarios finish within this many sprints — only 5% go beyond |
| **P50 / P75 / P95 overrun** | Extra sprints beyond plan at each percentile |
| **Cancelled %** | Share of scenarios where the feature was cancelled |

### What triggers a cancellation

A cancellation check fires when the simulated sprint count exceeds `planned sprints + max overrun sprints` (configured in `cancellation.max_sprints_over_plan`).
If the check fires, cancellation occurs with probability `cancellation.cancellation_probability`.

When a feature is cancelled:

- **Business value** is zero — the feature is not delivered.
- **Cost** is the spend already accrued up to the trigger point ($S_{\mathrm{plan}} + \Delta_{\max}$ sprints). This is the **sunk cost** — money already spent that cannot be recovered.

### The chart below

Shows the full sprint-count distribution per feature across all simulated scenarios — you can see how spread out delivery estimates are around the plan.

In [ ]:
# Sprint overrun summary — derived from delivery_results (Setup)
show.delay_summary(
    delivery_results,
    feature_names=feature_names,
)

# Exkurs: Sprint distribution chart
_ = plot_sprint_delay_grid(
    delivery_results,
    title="Delay Distribution — Actual vs Planned Sprints",
)

## 4) Cost Risk — Cost at Risk and CVaR

**Question:** How much could each feature cost beyond its planned budget — and what is the average cost in the worst scenarios?

### Plain language first

The planned investment is the approved budget. The simulation shows what delivery might actually cost when sprints run late.

- **Expected cost** — the average across all scenarios. Slightly above plan because overruns are more common than early delivery.
- **CaR 95%** — the cost you should be prepared for in all but the 5% worst scenarios.
- **CVaR 95%** — the *average* cost in those worst 5% scenarios (Expected Shortfall). This is what you budget for in a stressed scenario.

> **CaR** answers: *How bad can it get within the 95% confidence window?*
> **CVaR** answers: *When it breaks that limit, how bad is it on average?*

### How delivery cost is calculated (per scenario)

$$C_{\text{actual}} = S_{\text{actual}} \times L \times \frac{C_{\text{dev}}}{W}$$

More sprints → more team-weeks → more cost. The burn rate is constant — only sprint count varies.

### What the table shows

| Column | Meaning |
|---|---|
| **Planned investment** | $C_{\text{plan}}$ — the approved budget for this feature |
| **Expected cost** | Average delivery cost across all simulated scenarios |
| **Cost at Risk 95% (CaR)** | 95th percentile — only 5% of scenarios cost more than this |
| **CVaR (worst 5% avg)** | Average cost in the worst 5% of scenarios — the Expected Shortfall |
| **Uplift vs plan** | How much more than plan the expected cost is: $(E[C] / C_{\text{plan}}) - 1$ |
| **Cancelled %** | Share of scenarios where the feature was cancelled |

### The chart below

Compares planned vs expected (simulated) delivery cost per feature — a quick visual check of how much each feature's cost might drift from its approved budget.

In [ ]:
# Cost at Risk (CaR) & CVaR table — derived from delivery_results (Setup)
show.delivery_cost_risk(
    features,
    delivery_results,
)

# Exkurs: Cost comparison chart
_ = plot_cost_comparison(
    delivery_results,
    title="Planned vs Simulated Delivery Burn Cost",
)

## 5) Budget Fit — Which Features Stay Affordable?

**Question:** Under delivery risk, which features still fit the budget — and which combination has the lowest relative worst-case cost?

This is a **pure delivery cost view** — no business value, no business value. It answers two questions:

1. **Ranking** — which features have the least delivery cost pressure? Schedule these first.
2. **Cost Risk Selection** — which combination of features maximises portfolio size while minimising normalised CVaR 95%, within the scenario budget?

### Why CVaR 95% — not CaR 95%?

**CaR 95%** is a point estimate (the 95th percentile). Two feature subsets can have identical CaR 95% but very different tail behaviour beyond that point.

**CVaR 95%** (Expected Shortfall) is the *average* delivery cost across the worst 5% of scenarios. It captures the full severity of the tail — making it the right criterion when you want to minimise what delivery actually costs when things go wrong.

Sprint-based costs are also **step functions** — each extra sprint adds a fixed cost block. This causes the 95th percentile (CaR) to often land on the same multiple for every feature, making ranking meaningless. CVaR averages across the full tail and produces continuous, differentiating values.

### Why normalise CVaR by total planned cost?

Features have different durations and budgets. A combination of three short, cheap features will always have a lower **absolute** CVaR than a combination of three longer, more expensive features — simply because the total cost is smaller. That is not a risk difference, it is a scale difference.

Normalising removes the scale effect:

$$\text{Budget pressure} = \frac{\text{CVaR}_{95\%}^{\text{portfolio}}}{\text{total planned cost}} - 1$$

This asks: *by how much does the worst-case expected cost exceed what was planned for these features, proportionally?* The answer is comparable across any combination, regardless of runtime or budget size.

### Ranking table columns

| Column | Meaning |
|---|---|
| **Priority** | 1 = lowest budget pressure — safest to schedule first |
| **Planned investment** | $C_{\text{plan}}$ — the approved budget for this feature |
| **CVaR 95% (worst 5%)** | Average delivery cost in the worst 5% of sprint scenarios |
| **Budget pressure** | $\bigl(\text{CVaR}_{95\%} / C_{\text{plan}}\bigr) - 1$ — how far beyond plan in the tail |

### Budget summary

Shows the total planned investment vs the available scenario budget before any optimisation.

### Cost Risk Selection

The algorithm works in two steps:

1. **Maximise feature count** — find the largest number of features whose planned costs fit within the scenario budget.
2. **Minimise portfolio budget pressure** — among all combinations of that count, pick the one with the lowest $\text{CVaR}_{95\%} / \text{total planned cost} - 1$.

**Objective: include as many features as possible, then pick the combination with the lowest proportional worst-case cost overrun.**

$$\text{CVaR}_{95\%}^{\text{portfolio}} = E\!\left[\,\textstyle\sum_{i \in S} C_{\text{actual},i} \;\middle|\; \textstyle\sum_{i \in S} C_{\text{actual},i} \geq Q_{0.95}\right]$$

| Output | Meaning |
|---|---|
| **Selected features** | The largest subset fitting the budget, with the lowest normalised worst-case cost |
| **Total planned cost** | Sum of `development_cost` for selected features |
| **Budget remaining** | Unallocated scenario budget after selection |
| **Budget pressure** | $\text{CVaR}_{95\%} / \text{total planned cost} - 1$ — how far the worst-case cost exceeds what was planned for the selected features |
| **Expected delivery cost** | Mean portfolio delivery cost across all scenarios |
| **Portfolio CaR 95%** | 95th percentile of portfolio delivery cost |
| **Portfolio CVaR 95%** | Average portfolio delivery cost in the worst 5% — used to compute the minimised metric |

In [ ]:
show.budget_fit_ranking(features, delivery_results)

In [ ]:
show.budget_fit_and_select(features, delivery_results, float(scenario.budget))

---

## Learning Path

| Notebook | Topic |
|---|---|
| 01 | [Getting Started](01-getting-started.ipynb) |
| 02 | [Blockchain Case Study](02-blockchain-case-study.ipynb) |
| 03 | [Capital Budgeting](03-blockchain-case-study-capital-budgeting.ipynb) |
| 04 | [Portfolio Advisor](04-blockchain-case-study-advisor.ipynb) |
| 05 | [Risk Assessment](05-blockchain-case-study-risk.ipynb) |
| 06 | **Delivery Risk (sprint overruns, cost risk, budget fit)** ← You are here |
| 07 | [Executive Decision](07-blockchain-case-study-decision.ipynb) |
| A01 | [Portfolio Advisor](advanced/01-portfolio-advisor.ipynb) |
| A02 | [Portfolio Risk Dashboard](advanced/02-portfolio-risk-dashboard.ipynb) |

---

## Quick Reference

| Term | Definition |
|------|------------|
| **Sprint overrun** | When actual sprint count exceeds the planned sprint count for a feature. |
| **Sunk cost at cancellation** | Delivery cost already accrued up to the cancellation trigger point ($S_{\text{plan}} + \Delta_{\max}$ sprints). Remaining work is abandoned. |
| **Delivery Risk** | Probability that schedule overruns, cost variability, or cancellation causes delivery cost to exceed the planned investment. |
| **VaR / CaR 95%** | The cost threshold not exceeded in 95% of scenarios — a point estimate of worst-case cost. |
| **CVaR 95% (Expected Shortfall)** | Average delivery cost across the worst 5% of scenarios. Always ≥ CaR 95%. Coherent risk measure — captures full tail severity, not just the boundary. |
| **Feature budget pressure** | $\left(\text{CVaR}_{95\%} / C_{\text{plan}}\right) - 1$ — how far a feature's worst-case delivery cost exceeds its planned investment. |
| **Portfolio budget pressure** | $\text{CVaR}_{95\%}^{\text{portfolio}} / \text{total planned cost} - 1$ — how far the worst-case portfolio cost exceeds the total planned investment of the selected features. |
| **Portfolio CVaR 95%** | CVaR 95% of the summed per-scenario delivery costs across selected features. The metric minimised in Section 5. |
| **Lognormal distribution** | Distribution used to draw sprint delay factors. Long right tail: most scenarios finish near plan, a small fraction run significantly late. |

*Previous: [05-blockchain-case-study-risk.ipynb](05-blockchain-case-study-risk.ipynb) · Next: [07-blockchain-case-study-decision.ipynb](07-blockchain-case-study-decision.ipynb)*

---
**Previous:** [NB 05: Risk Analysis](05-blockchain-case-study-risk.ipynb) | **Next:** [NB 07: Executive Decision](07-blockchain-case-study-decision.ipynb)